[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/gps/gps.ipynb)

# Satellite Navigation

Satellite navigation locates a receiver and corrects its clock from the timestamps of arriving signals. Every transmission places the reception event on a light cone. Their intersection determines position and time together; uncertain timestamps spread that intersection into an uncertainty region whose shape depends on the satellite arrangement.

Projective spacetime algebra (STAP) holds events, light cones and measurement maps in `x+y+z+t-w0`. Its mixed signature supplies the causal geometry, and the null homogeneous direction carries translations. Light cones are quadratic correlations from events to planes; timing sensitivities build a separate positive information map whose inverse describes the uncertainty.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"
%matplotlib inline

In [ ]:
import numpy as np

from numga import Algebra, NumpyContext, stack, where
from examples.relativity.gps import render
from examples.util.animation import save_figure, show_animation

ga = Algebra("x+y+z+t-w0")
mv = NumpyContext(ga).multivector
Event = ga.gatype.antivector()
Slice = ga.gatype.vector()
Scalar = ga.gatype.scalar()
Readout = ga.gatype.from_blades("x y z t")
Step = Readout.dual().gatype.derive.output
Cone = ga.gatype((Slice, Event))
Information = ga.gatype((Readout, Step))
Covariance = ga.gatype((Step, Readout))
Orbit = ga.gatype.bivector()

origin = mv.w.dual()
across = mv.x.dual()
time = -mv.t.dual()
# The Lorentz form, reading a displacement's interval: the homogeneous direction squares to zero, so the
# complement carries it.
spacetime_metric = Event.dual_inverse().cast(Readout)

## Satellite signals

Six simulated satellites follow circular polar orbits at GPS altitude above a receiver at Earth's pole. Their timestamps have been corrected to GPS time; light propagation is modelled in flat spacetime. Lengths are in Earth radii and times in Earth-radius light-travel times, making the speed of light one. The orange surface marker is the receiver, and the blue segments connect it to the satellites at transmission.

In [ ]:
earth_radius_metres, orbit_altitude_metres = 6_371_000, 20_200_000
light_speed, gravitational_parameter = 299_792_458, 3.986004418e14
n_satellites, n_emission_steps = 6, 3
phase = np.deg2rad(20)
azimuths = np.arange(n_satellites) * (2 * np.pi / n_satellites)
phase_offsets = np.deg2rad([-12, -5, 8, 15, -9, 4])

# The planes share the polar axis; different phases give distinct elevations.
planes = (mv.yz * (-azimuths / 2)).exp() >> mv.xy
phases = mv.scalar((phase_offsets + phase)[..., None])
radius = (earth_radius_metres + orbit_altitude_metres) / earth_radius_metres
angular_speed = np.sqrt(gravitational_parameter / (earth_radius_metres + orbit_altitude_metres) ** 3)
angular_speed *= earth_radius_metres / light_speed

# Each snapshot uses its reception epoch as coordinate-time zero.
receiver = origin + across
initial_y, initial_z = 0.02, 0.01
coarse_position = receiver + mv.y.dual() * initial_y + mv.z.dual() * initial_z

In [ ]:
def orbit_positions(planes: Orbit, phases: Scalar, radius: float,
                    angular_speed: float, times: Scalar) -> Event:
    """Circular satellite trajectories, evaluated at coordinate times in the local inertial frame."""
    rotation = (-planes * ((phases + angular_speed * times) / 2)).exp()
    return origin + (rotation >> across) * radius + time * times


def emitted_events(receiver: Event, planes: Orbit, phases: Scalar, radius: float,
                   angular_speed: float, n_steps: int) -> Event:
    """Retarded transmission events on the orbits whose signals reach the receiver together."""
    reception_time = mv.t & receiver
    emission_time = reception_time
    for _ in range(n_steps):
        satellites = orbit_positions(planes, phases, radius, angular_speed, emission_time)
        separation = receiver - satellites
        # The separation's length in the receiver's space: its interval, with the time it spans added back.
        distance = ((separation & spacetime_metric(separation)) + (mv.t & separation).squared()).square_root()
        emission_time = reception_time - distance
    return orbit_positions(planes, phases, radius, angular_speed, emission_time)


emitted = emitted_events(receiver, planes, phases, radius, angular_speed, n_emission_steps)
save_figure(render.draw_constellation(emitted, receiver, radius), "gps_constellation")

## Intersecting light cones

At an event on the cone, `cone(event)` is its tangent plane and `event & cone(event)` vanishes. Translating the vertex carries this correlation through the translation and its adjugate. Two cones of one Lorentz form differ by a pair of planes, the plane at infinity and their radical plane, so the differences of four cones give three radical planes, which meet in a line, and the fourth cone cuts that line in two events. Only one of them comes after every transmission: the reception.

In [ ]:
def light_cones(emitted: Event) -> Cone:
    """Homogeneous quadratic correlations whose vertices are the transmission events."""
    centred = Event - (emitted - origin) * (mv.w & Event)
    # Carry the interval form through the translation on both its arguments.
    return centred.adjugate()(spacetime_metric(centred))


def intersect_cones(cones: Cone) -> Event:
    """The two intersections of four light cones, in a finite coordinate-time chart."""
    # Two cones of one Lorentz form differ by a pair of planes, the plane at infinity and their radical plane: the
    # difference sends an event to the radical plane times its weight plus the plane at infinity times the radical
    # plane's reading of it. Read at the origin, the second term is half the difference's own reading of the origin.
    difference = cones[1:] - cones[0]
    at_origin = difference(origin)
    radicals = at_origin - mv.w * ((origin & at_origin) / 2)
    axis = radicals[0] ^ radicals[1] ^ radicals[2]

    # Two time slices locate points on their common line. Normalizing by the
    # weight also gives these intersections their intrinsic event orientation.
    base = axis ^ mv.t
    base = base / (mv.w & base)
    tip = axis ^ (mv.t - mv.w)
    tip = tip / (mv.w & tip)
    direction = tip - base

    # The remaining cone cuts the line in two points.
    quadratic = direction & cones[0](direction)
    linear = base & cones[0](direction)
    constant = base & cones[0](base)
    root = (linear.squared() - quadratic * constant).square_root()
    parameters = stack((-linear - root, -linear + root)) / quadratic
    return base + direction * parameters


def reception(candidates: Event, emitted: Event) -> Event:
    """Of the cones' two intersections, the one every signal reaches after it left."""
    after = (mv.t & (candidates[:, None] - emitted)) > 0                  # [candidates, signals]
    return where(after[0].all(), candidates[0], candidates[1])


cones = light_cones(emitted[:4])
candidates = intersect_cones(cones)
located = reception(candidates, emitted[:4])
print(render.candidate_readout(candidates, coarse_position, earth_radius_metres, light_speed))

## Timing noise and clock error

Noisy signals need not meet at one event. The fit asks each signal's interval, `receiver & cone(receiver)`, to vanish, weighed against how far its timestamp's uncertainty moves that interval: twice the light-travel time. A signal's readout of a small displacement of the reception is twice its cone's tangent plane there, `cone(receiver)`, and contributes a positive dyad to `information`; the sum measures which displacements the satellites can distinguish. Solving against that map corrects the event, and `information.inverse()` gives its position-and-time covariance.

In [ ]:
range_error_metres, clock_error_seconds = 3, 30e-6
noise_seed, n_position_steps = 7, 5
positioning_extent_metres = 40
sigma = range_error_metres / earth_radius_metres
clock_reading = mv.scalar([clock_error_seconds * light_speed / earth_radius_metres])
noise = mv.scalar(np.random.default_rng(noise_seed).normal(size=(n_satellites, 1)) * sigma)

# Timestamp errors perturb the measured cone vertices in time.
observed = emitted + time * noise
initial = coarse_position + time * clock_reading

In [ ]:
def intervals(cones: Cone, emitted: Event, receiver: Event) -> tuple[Scalar, Readout, Scalar]:
    """Each signal's interval from its transmission to a reception event, `receiver & cone(receiver)`, which vanishes
    on the light cone; its readout of a small displacement of the reception, twice the cone's tangent plane there;
    and how far a timestamp's error moves it, twice the light-travel time."""
    tangent = cones(receiver[..., None])
    # The tangent plane's part across the plane at infinity reads no displacement.
    readout = 2 * tangent.cast(Readout)
    lever = 2 * (mv.t & (receiver[..., None] - emitted))
    return receiver[..., None] & tangent, readout, lever


def locate(emitted: Event, initial: Event, sigma: float,
           n_steps: int) -> tuple[Event, Information, Covariance]:
    """Fit the reception event to noisy timestamps, each to its light cone, and recover the local uncertainty map."""
    cones = light_cones(emitted)
    receiver = initial
    for _ in range(n_steps):
        interval, readout, lever = intervals(cones, emitted, receiver)
        # Each signal's readout contributes a positive cost for the displacements it can see, weighted by how far
        # its timestamp's uncertainty moves its interval.
        weight = 1 / (lever * sigma) ** 2
        information = (weight * readout * (readout & Step)).sum(axis=-1)
        mismatch = (weight * readout * interval).sum(axis=-1)
        receiver = receiver - information.solve(mismatch)

    _, readout, lever = intervals(cones, emitted, receiver)
    information = (readout * (readout & Step) / (lever * sigma) ** 2).sum(axis=-1)
    covariance = information.inverse()
    return receiver, information, covariance


estimate, information, covariance = locate(observed, initial, sigma, n_position_steps)
save_figure(render.draw_solution(emitted, receiver, estimate, covariance, radius,
                                 earth_radius_metres, positioning_extent_metres), "gps_positioning")
print(render.measurements(receiver, estimate, clock_reading, covariance, earth_radius_metres, light_speed))

The right panel magnifies the receiver's neighbourhood in metres. Orange marks the true position, blue the fitted position; the blue surface has principal radii equal to the position standard deviations. The clock uncertainty remains part of the inverse information map, so the position ellipsoid includes the difficulty of separating location from clock error.

## Satellite geometry and uncertainty

Satellites crowded into one patch of sky leave similar changes in range when the receiver moves radially. A clock error can imitate much of that change. Spreading the satellites across the sky separates those readouts and shrinks the uncertainty. Both panels keep their scales fixed, and each frame uses the same timestamp-noise draw. The Lorentz correlation sets the light cones throughout; the positive information correlation changes with the measurements.

In [ ]:
n_frames, duration_ms = 60, 65
uncertainty_extent_metres = 120
first_phase, last_phase = np.deg2rad([5, 50])


def geometry_frames(n_frames: int):
    for phase in np.linspace(first_phase, last_phase, n_frames):
        phases = mv.scalar((phase_offsets + phase)[..., None])
        emitted = emitted_events(receiver, planes, phases, radius, angular_speed, n_emission_steps)
        observed = emitted + time * noise
        estimate, _, covariance = locate(observed, initial, sigma, n_position_steps)
        yield emitted, receiver, estimate, covariance


show_animation(render.animate_geometry(geometry_frames(n_frames), radius, earth_radius_metres,
                                       uncertainty_extent_metres),
               "gps_uncertainty", duration_ms, n_frames=n_frames, forward_then_back=True)

In GPS positioning literature, [Bancroft's method](https://gssc.esa.int/navipedia/index.php/Bancroft_Method) gives an algebraic solution for four satellites, while [weighted least squares](https://gssc.esa.int/navipedia/index.php/Weighted_Least_Square_Solution_%28WLS%29) fits redundant measurements. The dependence of uncertainty on satellite arrangement is called [dilution of precision](https://gssc.esa.int/navipedia/index.php?title=Positioning_Error).